In [ ]:
import os
import gc
import json
import math
import time
import random
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")
TRAIN_MANIFEST = DATA_ROOT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = DATA_ROOT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST = DATA_ROOT / "manifests/test_manifest.jsonl"
TOKENIZER_PATH = DATA_ROOT / "tokenizer/legal_bpe_32k.json"

WORK_DIR = Path("/kaggle/working/phase1_rwkv")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from tokenizers import Tokenizer, decoders

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel() # Required to prevent Ġ / Ċ artifacts

VOCAB_SIZE = tokenizer.get_vocab_size()
print("Vocabulary:", VOCAB_SIZE)
assert VOCAB_SIZE == 32000

In [ ]:
# Reuse the existing WorkerSafeDocumentAwarePacker or LegalDataset from previous experiments
# Ensure standard causal-language-model labels (shifted).
CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

def load_manifest(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_manifest(TRAIN_MANIFEST)

class LegalDataset(torch.utils.data.Dataset):
    def __init__(self, records, tokenizer, context_length):
        self.records = records
        self.tokenizer = tokenizer
        self.context_length = context_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        text = self.records[idx].get("text", "")
        ids = self.tokenizer.encode(text).ids
        if len(ids) >= self.context_length + 1:
            ids = ids[:self.context_length + 1]
        else:
            ids = ids + [0] * ((self.context_length + 1) - len(ids))
        
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return {"input_ids": x, "labels": y}

train_dataset = LegalDataset(train_records, tokenizer, CONTEXT_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
print("DataLoader Ready. Context Length:", CONTEXT_LENGTH)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # Token shift interpolation parameters
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        # WKV exponential decay and bonus parameters
        self.time_decay = nn.Parameter(torch.zeros(d_model))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        # Linear projections
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
            num = torch.zeros(B, 1, D, device=x.device, dtype=torch.float32)
            den = torch.zeros(B, 1, D, device=x.device, dtype=torch.float32)
            p = torch.full((B, 1, D), -1e38, device=x.device, dtype=torch.float32)
        else:
            last_x, num, den, p = state

        # 1. Token Shift
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        v_x = x * self.time_mix_v + shifted_x * (1 - self.time_mix_v)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x)
        
        w = -torch.exp(self.time_decay.float())
        u = self.time_first.float()
        
        # 2. Vectorized WKV (No for-loop)
        # We unroll the recurrence into an exponential moving average in log-space
        # using PyTorch's cumulative operations. 
        # (This is an approximation of the strict sequential loop but leverages highly optimized backend ops)
        
        time_curve = torch.arange(T, device=x.device, dtype=torch.float32).view(1, T, 1) * w
        
        # Calculate shifted Keys
        k_shifted = k + time_curve
        
        # Compute exponents safely
        p_max = torch.max(k_shifted, dim=1, keepdim=True)[0]
        exp_k = torch.exp(k_shifted - p_max)
        
        # Cumulative sums over time
        num_c = torch.cumsum(exp_k * v, dim=1)
        den_c = torch.cumsum(exp_k, dim=1)
        
        # Add the 'u' bonus for the current token
        wkv = (num_c + torch.exp(k + u - p_max) * v) / (den_c + torch.exp(k + u - p_max))
        
        # 3. Output
        rwkv = torch.sigmoid(r) * wkv.to(x.dtype)
        out = self.output(rwkv)
        
        # Fake state return to satisfy interface (for stateless training)
        return out, (x[:, -1:, :], num, den, p)


class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # Token shift interpolation parameters
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        # In RWKV, Channel Mixing acts as the Feed-Forward layer, usually expanding by 4x
        hidden_size = 4 * d_model
        
        self.key = nn.Linear(d_model, hidden_size, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(hidden_size, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
        else:
            last_x = state
            
        # Token shift
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        # Key projection + Squared ReLU activation
        k = self.key(k_x)
        k = torch.relu(k) ** 2
        
        # Value projection
        v = self.value(k)
        
        # Receptance gate
        r = self.receptance(r_x)
        out = torch.sigmoid(r) * v
        
        new_state = x[:, -1:, :]
        return out, new_state


class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = RWKVChannelMixing(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, states=None):
        # Residual connections around time and channel mixing
        x_tm, new_tm_state = self.time_mix(self.ln1(x), None if states is None else states[0])
        x = x + x_tm
        
        x_cm, new_cm_state = self.channel_mix(self.ln2(x), None if states is None else states[1])
        x = x + x_cm
        
        return x, (new_tm_state, new_cm_state)


class RWKVLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model) for _ in range(num_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        
        for block in self.blocks:
            x, _ = block(x)
            
        x = self.ln_out(x)
        logits = self.head(x)
        
        loss = None
        if labels is not None:
            # Shifted labels for cross entropy
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
        return logits, loss

model = RWKVLanguageModel().to(DEVICE)

In [ ]:
# Rebuild model after replacing forward()
model = RWKVLanguageModel().to(DEVICE)

# Parameter count
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# 4096-token test
x = torch.randint(0, 32000, (1, 4096), device=DEVICE)
y = torch.randint(0, 32000, (1, 4096), device=DEVICE)

torch.cuda.synchronize()
start = time.time()

logits, loss = model(x, y)

torch.cuda.synchronize()
print("Forward:", time.time() - start)

start = time.time()
loss.backward()

torch.cuda.synchronize()
print("Backward:", time.time() - start)
print("Loss:", loss.item())

In [ ]:
# ============================================================
# CELL 8 — VECTORIZED RWKV ARCHITECTURE
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        self.time_decay = nn.Parameter(torch.zeros(d_model))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
            num = torch.zeros(B, 1, D, device=x.device, dtype=torch.float32)
            den = torch.zeros(B, 1, D, device=x.device, dtype=torch.float32)
            p = torch.full((B, 1, D), -1e38, device=x.device, dtype=torch.float32)
        else:
            last_x, num, den, p = state

        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        v_x = x * self.time_mix_v + shifted_x * (1 - self.time_mix_v)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x)
        
        w = -torch.exp(self.time_decay.float())
        u = self.time_first.float()
        
        time_curve = torch.arange(T, device=x.device, dtype=torch.float32).view(1, T, 1) * w
        k_shifted = k + time_curve
        p_max = torch.max(k_shifted, dim=1, keepdim=True)[0]
        exp_k = torch.exp(k_shifted - p_max)
        
        num_c = torch.cumsum(exp_k * v, dim=1)
        den_c = torch.cumsum(exp_k, dim=1)
        
        wkv = (num_c + torch.exp(k + u - p_max) * v) / (den_c + torch.exp(k + u - p_max))
        rwkv = torch.sigmoid(r) * wkv.to(x.dtype)
        out = self.output(rwkv)
        
        return out, (x[:, -1:, :], num, den, p)

class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        hidden_size = 4 * d_model
        self.key = nn.Linear(d_model, hidden_size, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(hidden_size, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype) if state is None else state
            
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x)
        k = torch.relu(k) ** 2
        v = self.value(k)
        r = self.receptance(r_x)
        out = torch.sigmoid(r) * v
        
        return out, x[:, -1:, :]

class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = RWKVChannelMixing(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, states=None):
        x_tm, new_tm_state = self.time_mix(self.ln1(x), None if states is None else states[0])
        x = x + x_tm
        x_cm, new_cm_state = self.channel_mix(self.ln2(x), None if states is None else states[1])
        x = x + x_cm
        return x, (new_tm_state, new_cm_state)

class RWKVLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model) for _ in range(num_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        x = self.ln_out(x)
        logits = self.head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
        return logits, loss

model = RWKVLanguageModel().to(DEVICE)
print("✓ Vectorized RWKV Model Initialized")

In [ ]:
# ============================================================
# CELL 11 — MECHANICAL VALIDATION
# ============================================================
import math

model.train()
optimizer.zero_grad(set_to_none=True)

# 1. Forward Pass
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    logits, loss = model(input_ids, labels=labels)

assert torch.isfinite(loss), "Validation Failed: Loss is not finite."
print(f"✓ Forward pass finite (Loss: {loss.item():.4f})")

# 2. Backward Pass
scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
assert torch.isfinite(grad_norm), "Validation Failed: Gradient norm is not finite."
print(f"✓ Backward pass finite (Grad Norm: {grad_norm.item():.4f})")

# 3. NaN/Inf Check in Gradients
has_nans = False
for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        print(f"NaN/Inf detected in gradient of: {name}")
        has_nans = True
assert not has_nans, "Validation Failed: NaNs/Infs found in gradients."
print("✓ No NaNs or Infs in gradients")

# 4. Optimizer Step & Memory
scaler.step(optimizer)
scaler.update()
print("✓ Optimizer step successful")

mem_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"✓ Stable Memory Allocated: {mem_alloc:.2f} GB")

# 5. Checkpoint Verification
save_checkpoint(0, reason="validation_test")
print("\nALL MECHANICAL TESTS PASSED. READY FOR TRAINING.")

In [ ]:
# ============================================================
# CELL 8 — STABLE JIT-COMPILED RWKV ARCHITECTURE
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Compile the exact, numerically stable WKV loop to C++ TorchScript
@torch.jit.script
def wkv_forward_jit(
    k: torch.Tensor, v: torch.Tensor, r: torch.Tensor, 
    time_decay: torch.Tensor, time_first: torch.Tensor, 
    num: torch.Tensor, den: torch.Tensor, p: torch.Tensor
):
    B: int = k.size(0)
    T: int = k.size(1)
    
    out = torch.empty_like(v)
    w = -torch.exp(time_decay)
    u = time_first
    
    for t in range(T):
        kt = k[:, t, :]
        vt = v[:, t, :]
        rt = r[:, t, :]
        
        # Current token WKV calculation (guaranteed e1 or e2 is 1.0, avoiding 0/0)
        p_curr = torch.maximum(p, kt + u)
        e1 = torch.exp(p - p_curr)
        e2 = torch.exp(kt + u - p_curr)
        wkv = (e1 * num + e2 * vt) / (e1 * den + e2)
        out[:, t, :] = torch.sigmoid(rt) * wkv
        
        # Update running state for the next token
        p_next = torch.maximum(p + w, kt)
        e1_next = torch.exp(p + w - p_next)
        e2_next = torch.exp(kt - p_next)
        
        num = e1_next * num + e2_next * vt
        den = e1_next * den + e2_next
        p = p_next
        
    return out, num, den, p

# 2. Time Mixing using the JIT WKV
class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        self.time_decay = nn.Parameter(torch.zeros(d_model))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        # Unpack state or initialize with safe bounds
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
            num = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            den = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            p = torch.full((B, D), -1e38, device=x.device, dtype=torch.float32)
        else:
            last_x, num, den, p = state

        # Interpolate with previous token
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        v_x = x * self.time_mix_v + shifted_x * (1 - self.time_mix_v)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        # Project inputs (WKV requires float32 computation)
        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x).float()
        
        # Execute C++ TorchScript loop
        rwkv, num, den, p = wkv_forward_jit(
            k, v, r, 
            self.time_decay.float(), 
            self.time_first.float(), 
            num, den, p
        )
        
        out = self.output(rwkv.to(x.dtype))
        return out, (x[:, -1:, :], num, den, p)

class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        hidden_size = 4 * d_model
        self.key = nn.Linear(d_model, hidden_size, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(hidden_size, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype) if state is None else state[0]
            
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x)
        k = torch.relu(k) ** 2
        v = self.value(k)
        r = self.receptance(r_x)
        out = torch.sigmoid(r) * v
        
        return out, (x[:, -1:, :], )

class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = RWKVChannelMixing(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, states=None):
        x_tm, new_tm_state = self.time_mix(self.ln1(x), None if states is None else states[0])
        x = x + x_tm
        x_cm, new_cm_state = self.channel_mix(self.ln2(x), None if states is None else states[1])
        x = x + x_cm
        return x, (new_tm_state, new_cm_state)

class RWKVLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model) for _ in range(num_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        x = self.ln_out(x)
        logits = self.head(x)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
        return logits, loss

model = RWKVLanguageModel().to(DEVICE)
print(f"✓ Stable JIT RWKV Model Initialized ({sum(p.numel() for p in model.parameters()):,} parameters)")

In [ ]:
# ============================================================
# CELL 11 — MECHANICAL VALIDATION (Self-Contained State Reset)
# ============================================================
import math
import torch

# 0. Re-declare hyperparams to ensure they exist in this cell's scope
LR = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

# Reset optimizer and scaler to clear out the crashed state
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

model.train()
optimizer.zero_grad(set_to_none=True)

# 1. Forward Pass
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    logits, loss = model(input_ids, labels=labels)

assert torch.isfinite(loss), "Validation Failed: Loss is not finite."
print(f"✓ Forward pass finite (Loss: {loss.item():.4f})")

# 2. Backward Pass
scaler.scale(loss).backward()
scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
assert torch.isfinite(grad_norm), "Validation Failed: Gradient norm is not finite."
print(f"✓ Backward pass finite (Grad Norm: {grad_norm.item():.4f})")

# 3. NaN/Inf Check in Gradients
has_nans = False
for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        print(f"NaN/Inf detected in gradient of: {name}")
        has_nans = True
assert not has_nans, "Validation Failed: NaNs/Infs found in gradients."
print("✓ No NaNs or Infs in gradients")

# 4. Optimizer Step & Memory
scaler.step(optimizer)
scaler.update()
print("✓ Optimizer step successful")

mem_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"✓ Stable Memory Allocated: {mem_alloc:.2f} GB")

print("\nALL MECHANICAL TESTS PASSED. READY FOR TRAINING.")

In [ ]:
# ============================================================
# CELL 8 — FP16-SAFEGUARDED RWKV ARCHITECTURE
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

@torch.jit.script
def wkv_forward_jit(
    k: torch.Tensor, v: torch.Tensor, r: torch.Tensor, 
    time_decay: torch.Tensor, time_first: torch.Tensor, 
    num: torch.Tensor, den: torch.Tensor, p: torch.Tensor
):
    B: int = k.size(0)
    T: int = k.size(1)
    
    out = torch.empty_like(v)
    w = -torch.exp(time_decay)
    u = time_first
    
    for t in range(T):
        kt = k[:, t, :]
        vt = v[:, t, :]
        rt = r[:, t, :]
        
        p_curr = torch.maximum(p, kt + u)
        e1 = torch.exp(p - p_curr)
        e2 = torch.exp(kt + u - p_curr)
        
        den_out = torch.clamp(e1 * den + e2, min=1e-8)
        wkv = (e1 * num + e2 * vt) / den_out
        
        out[:, t, :] = torch.sigmoid(rt) * wkv
        
        p_next = torch.maximum(p + w, kt)
        e1_next = torch.exp(p + w - p_next)
        e2_next = torch.exp(kt - p_next)
        
        num = e1_next * num + e2_next * vt
        den = e1_next * den + e2_next
        p = p_next
        
    return out, num, den, p

class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        self.time_decay = nn.Parameter(torch.full((d_model,), -2.0))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
            num = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            den = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            p = torch.full((B, D), -1e38, device=x.device, dtype=torch.float32)
        else:
            last_x, num, den, p = state

        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        v_x = x * self.time_mix_v + shifted_x * (1 - self.time_mix_v)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x).float()
        
        rwkv, num, den, p = wkv_forward_jit(
            k, v, r, 
            self.time_decay.float(), 
            self.time_first.float(), 
            num, den, p
        )
        
        out = self.output(rwkv.to(x.dtype))
        return out, (x[:, -1:, :], num, den, p)

class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        hidden_size = 4 * d_model
        self.key = nn.Linear(d_model, hidden_size, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(hidden_size, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype) if state is None else state[0]
            
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x)
        
        # SAFEGUARD: Force float32, square, and CLAMP to prevent FP16 overflow
        k = torch.relu(k.float()) ** 2
        k = torch.clamp(k, max=65500.0) 
        
        v = self.value(k.to(x.dtype))
        r = self.receptance(r_x)
        out = torch.sigmoid(r) * v
        
        return out, (x[:, -1:, :], )

class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = RWKVChannelMixing(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, states=None):
        x_tm, new_tm_state = self.time_mix(self.ln1(x), None if states is None else states[0])
        x = x + x_tm
        x_cm, new_cm_state = self.channel_mix(self.ln2(x), None if states is None else states[1])
        x = x + x_cm
        return x, (new_tm_state, new_cm_state)

class RWKVLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model) for _ in range(num_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        x = self.ln_out(x)
        logits = self.head(x)
        
        loss = None
        if labels is not None:
            # SAFEGUARD: Cast logits to float32 before cross entropy
            shift_logits = logits[..., :-1, :].contiguous().float()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
        return logits, loss

# Instantiate the final safeguarded model
model = RWKVLanguageModel().to(DEVICE)
print(f"✓ Fully Safeguarded RWKV Model Initialized ({sum(p.numel() for p in model.parameters()):,} parameters)")

In [ ]:
# ============================================================
# CELL 11 — MECHANICAL VALIDATION (Diagnostic Version)
# ============================================================
import math
import torch

LR = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

# 0. Clean reset
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

model.train()
optimizer.zero_grad(set_to_none=True)

# 1. Forward Pass
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    logits, loss = model(input_ids, labels=labels)

assert torch.isfinite(loss), "Validation Failed: Loss is not finite."
print(f"✓ Forward pass finite (Loss: {loss.item():.4f})")

# 2. Backward Pass
scaler.scale(loss).backward()
scaler.unscale_(optimizer)

bad = []

for name, p in model.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append({
                "name": name,
                "grad_max": p.grad.detach().abs().max().item(),
                "grad_nan": torch.isnan(p.grad).any().item(),
                "grad_inf": torch.isinf(p.grad).any().item(),
            })

print("BAD GRADIENTS:")
for x in bad:
    print(x)

# 3. Detect NaNs *before* checking the norm to isolate the culprit
has_nans = False
for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        print(f"NaN/Inf detected in gradient of: {name}")
        has_nans = True
        
assert not has_nans, "Validation Failed: NaNs/Infs found in gradients!"
print("✓ No NaNs or Infs in gradients")

# 4. Clip & Step
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
assert torch.isfinite(grad_norm), "Validation Failed: Gradient norm is not finite."
print(f"✓ Backward pass finite (Grad Norm: {grad_norm.item():.4f})")

scaler.step(optimizer)
scaler.update()
print("✓ Optimizer step successful")

mem_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"✓ Stable Memory Allocated: {mem_alloc:.2f} GB")
print("\nALL MECHANICAL TESTS PASSED. READY FOR TRAINING.")

In [ ]:
# ============================================================
# CELL 8 — RWKV ARCHITECTURE WITH FP32 HEAD BOUNDARY
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

@torch.jit.script
def wkv_forward_jit(
    k: torch.Tensor, v: torch.Tensor, r: torch.Tensor, 
    time_decay: torch.Tensor, time_first: torch.Tensor, 
    num: torch.Tensor, den: torch.Tensor, p: torch.Tensor
):
    B: int = k.size(0)
    T: int = k.size(1)
    
    out = torch.empty_like(v)
    w = -torch.exp(time_decay)
    u = time_first
    
    for t in range(T):
        kt = k[:, t, :]
        vt = v[:, t, :]
        rt = r[:, t, :]
        
        p_curr = torch.maximum(p, kt + u)
        e1 = torch.exp(p - p_curr)
        e2 = torch.exp(kt + u - p_curr)
        
        den_out = torch.clamp(e1 * den + e2, min=1e-8)
        wkv = (e1 * num + e2 * vt) / den_out
        
        out[:, t, :] = torch.sigmoid(rt) * wkv
        
        p_next = torch.maximum(p + w, kt)
        e1_next = torch.exp(p + w - p_next)
        e2_next = torch.exp(kt - p_next)
        
        num = e1_next * num + e2_next * vt
        den = e1_next * den + e2_next
        p = p_next
        
    return out, num, den, p

class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        self.time_decay = nn.Parameter(torch.full((d_model,), -2.0))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        
        if state is None:
            last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype)
            num = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            den = torch.zeros(B, D, device=x.device, dtype=torch.float32)
            p = torch.full((B, D), -1e38, device=x.device, dtype=torch.float32)
        else:
            last_x, num, den, p = state

        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        v_x = x * self.time_mix_v + shifted_x * (1 - self.time_mix_v)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x).float()
        
        rwkv, num, den, p = wkv_forward_jit(
            k, v, r, 
            self.time_decay.float(), 
            self.time_first.float(), 
            num, den, p
        )
        
        out = self.output(rwkv.to(x.dtype))
        return out, (x[:, -1:, :], num, den, p)

class RWKVChannelMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        
        hidden_size = 4 * d_model
        self.key = nn.Linear(d_model, hidden_size, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(hidden_size, d_model, bias=False)
        
    def forward(self, x, state=None):
        B, T, D = x.size()
        last_x = torch.zeros(B, 1, D, device=x.device, dtype=x.dtype) if state is None else state[0]
            
        shifted_x = torch.cat([last_x, x[:, :-1, :]], dim=1)
        k_x = x * self.time_mix_k + shifted_x * (1 - self.time_mix_k)
        r_x = x * self.time_mix_r + shifted_x * (1 - self.time_mix_r)
        
        k = self.key(k_x)
        k = torch.relu(k.float()) ** 2
        k = torch.clamp(k, max=65500.0) 
        
        v = self.value(k.to(x.dtype))
        r = self.receptance(r_x)
        out = torch.sigmoid(r) * v
        
        return out, (x[:, -1:, :], )

class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = RWKVChannelMixing(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
    def forward(self, x, states=None):
        x_tm, new_tm_state = self.time_mix(self.ln1(x), None if states is None else states[0])
        x = x + x_tm
        x_cm, new_cm_state = self.channel_mix(self.ln2(x), None if states is None else states[1])
        x = x + x_cm
        return x, (new_tm_state, new_cm_state)

class RWKVLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, num_layers=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model) for _ in range(num_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x, _ = block(x)
        x = self.ln_out(x)
        
        # Explicit FP32 boundary and autocast disable
        with torch.amp.autocast("cuda", enabled=False):
            logits = self.head(x.float())
            loss = None
            if labels is not None:
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()
                loss = F.cross_entropy(
                    shift_logits.view(-1, shift_logits.size(-1)),
                    shift_labels.view(-1)
                )
                
        return logits, loss

model = RWKVLanguageModel().to(DEVICE)
model.head.float() # Keep final projection in FP32
print(f"✓ Model Initialized with FP32 Head ({sum(p.numel() for p in model.parameters()):,} parameters)")

In [ ]:
# ============================================================
# CELL 11 — MECHANICAL VALIDATION
# ============================================================
import math
import torch

LR = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

model.train()
optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

# 1. Forward Pass
with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    logits, loss = model(input_ids, labels=labels)

assert torch.isfinite(loss), "Validation Failed: Loss is not finite."
print(f"✓ Forward pass finite (Loss: {loss.item():.4f})")

# 2. Backward Pass
scaler.scale(loss).backward()
scaler.unscale_(optimizer)

has_nans = False
for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        print(f"NaN/Inf detected in gradient of: {name}")
        has_nans = True
        
assert not has_nans, "Validation Failed: NaNs/Infs found in gradients!"
print("✓ No NaNs or Infs in gradients")

# 3. Clip & Step
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
assert torch.isfinite(grad_norm), "Validation Failed: Gradient norm is not finite."
print(f"✓ Backward pass finite (Grad Norm: {grad_norm.item():.4f})")

scaler.step(optimizer)
scaler.update()
print("✓ Optimizer step successful")

print("\nALL MECHANICAL TESTS PASSED.")

In [ ]:
# ============================================================
# TIMING BENCHMARK (Self-Contained)
# ============================================================
import time
import torch

GRADIENT_ACCUMULATION_STEPS = 8
MAX_GRAD_NORM = 1.0

model.train()
torch.cuda.synchronize()
start_time = time.time()

optimizer.zero_grad(set_to_none=True)
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    logits, loss = model(input_ids=input_ids, labels=labels)
    scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

scaler.scale(scaled_loss).backward()
scaler.unscale_(optimizer)
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
scaler.step(optimizer)
scaler.update()

torch.cuda.synchronize()
elapsed = time.time() - start_time

print(f"Single micro-step runtime: {elapsed:.3f} seconds")
print(f"Estimated time for 1 full optimizer step (8 micro-steps): {elapsed * 8:.2f} seconds")
print(f"Estimated time for full 1000-step training: {(elapsed * 8 * 1000) / 3600:.2f} hours")

In [ ]:
del logits, loss, input_ids, labels
torch.cuda.empty_cache()

In [ ]:
import gc
import torch
import time

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model = RWKVLanguageModel(
    vocab_size=32000,
    d_model=768,
    num_layers=12
).to(DEVICE)

# ONLY the final vocabulary projection stays FP32
model.head.float()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Head dtype: {model.head.weight.dtype}")

In [ ]:
def forward(self, input_ids, labels=None):
    x = self.embedding(input_ids)

    for block in self.blocks:
        x, _ = block(x)

    x = self.ln_out(x)

    if labels is None:
        # Inference path
        with torch.autocast(
            device_type="cuda",
            enabled=False
        ):
            logits = self.head(x.float())
        return logits, None

    x_flat = x[:, :-1, :].contiguous()
    y_flat = labels[:, 1:].contiguous()

    B, T, D = x_flat.shape

    x_flat = x_flat.view(-1, D)
    y_flat = y_flat.view(-1)

    CHUNK = 256

    total_loss = torch.zeros(
        (),
        device=x.device,
        dtype=torch.float32
    )

    total_tokens = 0

    # FP32 vocabulary projection, chunk by chunk
    for start in range(0, B * T, CHUNK):
        end = min(start + CHUNK, B * T)

        h = x_flat[start:end].float()
        target = y_flat[start:end]

        logits_chunk = self.head(h)

        chunk_loss = F.cross_entropy(
            logits_chunk,
            target,
            reduction="sum"
        )

        total_loss = total_loss + chunk_loss
        total_tokens += end - start

    loss = total_loss / total_tokens

    return None, loss

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

with torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
    enabled=USE_AMP
):
    _, loss = model(
        input_ids=input_ids,
        labels=labels
    )

print(f"Loss: {loss.item():.4f}")
print(f"Loss finite: {torch.isfinite(loss).item()}")

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

bad = []

for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        bad.append(name)

print("Bad gradients:", bad)

print(
    f"Peak memory: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

if len(bad) == 0:
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    print(f"Grad norm: {grad_norm.item():.4f}")
    print("✓ BACKWARD PASSED")
else:
    print("✗ Gradient overflow remains")

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)

# No autocast
_, loss = model(
    input_ids=input_ids,
    labels=labels
)

print("Loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

loss.backward()

bad = []

for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        bad.append(name)

print("Bad gradients:", bad)

head_grad = model.head.weight.grad

if head_grad is not None:
    print(
        "Head grad max:",
        head_grad.detach().abs().max().item()
    )

print(
    "Peak memory:",
    torch.cuda.max_memory_allocated() / 1024**3,
    "GB"
)

In [ ]:
model.zero_grad(set_to_none=True)

with torch.no_grad():
    x = model.embedding(input_ids)

    for i, block in enumerate(model.blocks):
        x, _ = block(x)

        print(
            f"Block {i:02d}: "
            f"finite={torch.isfinite(x).all().item()} "
            f"max={x.abs().max().item():.4e}"
        )

    x = model.ln_out(x)

    print(
        "After LN:",
        x.abs().max().item(),
        "finite:",
        torch.isfinite(x).all().item()
    )

    # Only inspect a small chunk of the vocabulary projection
    h = x[:, :-1, :].reshape(-1, x.size(-1))[:256]

    logits_test = model.head(h)

    print(
        "Logits:",
        "max=", logits_test.abs().max().item(),
        "finite=", torch.isfinite(logits_test).all().item()
    )

In [ ]:
model.zero_grad(set_to_none=True)

with torch.no_grad():
    x = model.embedding(input_ids)

    for i, block in enumerate(model.blocks):
        x, _ = block(x)

        print(
            f"Block {i:02d}: "
            f"finite={torch.isfinite(x).all().item()} "
            f"max={x.abs().max().item():.4e}"
        )

    x = model.ln_out(x)

    print(
        "After LN:",
        x.abs().max().item(),
        "finite:",
        torch.isfinite(x).all().item()
    )

    # Only inspect a small chunk of the vocabulary projection
    h = x[:, :-1, :].reshape(-1, x.size(-1))[:256]

    logits_test = model.head(h)

    print(
        "Logits:",
        "max=", logits_test.abs().max().item(),
        "finite=", torch.isfinite(logits_test).all().item()
    )

In [ ]:
import gc
import torch
import time

gc.collect()
torch.cuda.empty_cache()

# Make the entire model FP32
model = model.float()

# Recreate optimizer after dtype conversion
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

model.train()
optimizer.zero_grad(set_to_none=True)

torch.cuda.reset_peak_memory_stats()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

torch.cuda.synchronize()
start = time.time()

# NO autocast
logits, loss = model(
    input_ids=input_ids,
    labels=labels
)

print(f"Loss: {loss.item():.4f}")

loss.backward()

bad = []

for name, p in model.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        bad.append(name)

print("Bad gradients:", bad)

assert len(bad) == 0, f"Bad gradients: {bad}"

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    1.0
)

optimizer.step()

torch.cuda.synchronize()
elapsed = time.time() - start

print(f"Grad norm: {grad_norm.item():.4f}")
print(f"One FP32 training step: {elapsed:.3f} sec")
print(
    f"Peak memory: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)
print("✓ FP32 optimizer step completed")

In [ ]:
class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))

        self.time_decay = nn.Parameter(
            torch.full((d_model,), -2.0)
        )
        self.time_first = nn.Parameter(
            torch.zeros(d_model)
        )

        self.key = nn.Linear(
            d_model, d_model, bias=False
        )
        self.value = nn.Linear(
            d_model, d_model, bias=False
        )
        self.receptance = nn.Linear(
            d_model, d_model, bias=False
        )
        self.output = nn.Linear(
            d_model, d_model, bias=False
        )

    def forward(self, x, state=None):
        B, T, D = x.size()

        if state is None:
            last_x = torch.zeros(
                B, 1, D,
                device=x.device,
                dtype=x.dtype
            )

            num = torch.zeros(
                B, D,
                device=x.device,
                dtype=torch.float32
            )

            den = torch.zeros(
                B, D,
                device=x.device,
                dtype=torch.float32
            )

            p = torch.full(
                (B, D),
                -1e38,
                device=x.device,
                dtype=torch.float32
            )

        else:
            last_x, num, den, p = state

        # Token shift
        shifted_x = torch.cat(
            [last_x, x[:, :-1, :]],
            dim=1
        )

        k_x = (
            x * self.time_mix_k
            + shifted_x * (1 - self.time_mix_k)
        )

        v_x = (
            x * self.time_mix_v
            + shifted_x * (1 - self.time_mix_v)
        )

        r_x = (
            x * self.time_mix_r
            + shifted_x * (1 - self.time_mix_r)
        )

        k = self.key(k_x).float()
        v = self.value(v_x).float()
        r = self.receptance(r_x).float()

        w = -torch.exp(
            self.time_decay.float()
        )

        u = self.time_first.float()

        # Vectorized WKV
        time_curve = (
            torch.arange(
                T,
                device=x.device,
                dtype=torch.float32
            ).view(1, T, 1)
            * w.view(1, 1, D)
        )

        k_shifted = k + time_curve

        p_max = torch.max(
            k_shifted,
            dim=1,
            keepdim=True
        )[0]

        exp_k = torch.exp(
            k_shifted - p_max
        )

        num_c = torch.cumsum(
            exp_k * v,
            dim=1
        )

        den_c = torch.cumsum(
            exp_k,
            dim=1
        )

        bonus = torch.exp(
            k + u.view(1, 1, D) - p_max
        )

        wkv = (
            num_c + bonus * v
        ) / (
            den_c + bonus
        )

        rwkv = (
            torch.sigmoid(r)
            * wkv.to(x.dtype)
        )

        out = self.output(rwkv)

        return out, (
            x[:, -1:, :],
            num,
            den,
            p
        )

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model = RWKVLanguageModel(
    vocab_size=32000,
    d_model=768,
    num_layers=12
).to(DEVICE).float()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    "GPU memory:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()
optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

torch.cuda.synchronize()
start = time.time()

# IMPORTANT: no autocast
_, loss = model(
    input_ids=input_ids,
    labels=labels
)

print(f"Loss: {loss.item():.4f}")
print(
    "Loss finite:",
    torch.isfinite(loss).item()
)

loss.backward()

bad = []

for name, p in model.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print("Bad gradients:", bad)

assert torch.isfinite(loss)
assert len(bad) == 0, f"Bad gradients: {bad}"

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    1.0
)

optimizer.step()

torch.cuda.synchronize()
elapsed = time.time() - start

print(f"Grad norm: {grad_norm.item():.4f}")
print(f"Runtime: {elapsed:.3f} sec")
print(
    "Peak memory:",
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

print("✓ Vectorized FP32 training step passed")

In [ ]:
del input_ids, labels, batch
optimizer.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU after cleanup:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

In [ ]:
import gc
import time
import torch

gc.collect()
torch.cuda.empty_cache()

# Rebuild clean model
model = RWKVLanguageModel(
    vocab_size=32000,
    d_model=768,
    num_layers=12
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

model.train()
optimizer.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

torch.cuda.synchronize()
start = time.time()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    _, loss = model(
        input_ids=input_ids,
        labels=labels
    )

print(f"Loss: {loss.item():.4f}")
print("Loss finite:", torch.isfinite(loss).item())

scaler.scale(loss).backward()

# IMPORTANT: unscale before checking gradients
scaler.unscale_(optimizer)

bad = []

for name, p in model.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print("Bad gradients:", bad)

if len(bad) == 0:
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    scaler.step(optimizer)
    scaler.update()

    torch.cuda.synchronize()
    elapsed = time.time() - start

    print(f"Grad norm: {grad_norm.item():.4f}")
    print(f"Runtime: {elapsed:.3f} sec")
    print(
        "Peak memory:",
        f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
    )
    print("✓ AMP TRAINING STEP PASSED")

else:
    print("✗ AMP gradient overflow")

In [ ]:
import gc
import time
import torch

gc.collect()
torch.cuda.empty_cache()

# Make sure we're using FP32
model = model.float()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

model.train()
optimizer.zero_grad(set_to_none=True)

ACCUM = 8

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

torch.cuda.synchronize()
start = time.time()

for micro in range(ACCUM):

    batch = next(iter(train_loader))

    input_ids = batch["input_ids"].to(
        DEVICE,
        non_blocking=True
    )

    labels = batch["labels"].to(
        DEVICE,
        non_blocking=True
    )

    _, loss = model(
        input_ids=input_ids,
        labels=labels
    )

    loss = loss / ACCUM

    loss.backward()

    del input_ids, labels, batch

# Check gradients
bad = []

for name, p in model.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print("Bad gradients:", bad)

assert len(bad) == 0, f"Bad gradients: {bad}"

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    1.0
)

optimizer.step()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()
elapsed = time.time() - start

print()
print("=" * 50)
print("FP32 RWKV BENCHMARK")
print("=" * 50)
print(f"8 micro-steps:       {elapsed:.2f} sec")
print(f"1 optimizer step:     {elapsed:.2f} sec")
print(f"1000 optimizer steps: {elapsed * 1000 / 3600:.2f} hours")
print(f"Grad norm:            {grad_norm.item():.4f}")
print(
    f"Peak GPU memory:      "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)
print("=" * 50)

In [ ]:
# ============================================================
# RWKV TRAINING — 1000 OPTIMIZER STEPS
# ============================================================

import os
import time
import gc
import torch

MAX_STEPS = 1000
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0

CHECKPOINT_DIR = "/kaggle/working/phase1_rwkv"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_latest.pt"
)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

# ------------------------------------------------------------
# Resume if checkpoint exists
# ------------------------------------------------------------

start_step = 0

if os.path.exists(CHECKPOINT_PATH):

    print("Loading checkpoint...")

    ckpt = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer_state_dict"]
    )

    start_step = ckpt["step"]

    print(
        f"✓ Resumed from optimizer step "
        f"{start_step}"
    )

else:
    print("Starting training from step 0")


# ------------------------------------------------------------
# Training iterator
# ------------------------------------------------------------

train_iter = iter(train_loader)

def get_batch():
    global train_iter

    try:
        batch = next(train_iter)

    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    return batch


# ------------------------------------------------------------
# Checkpoint function
# ------------------------------------------------------------

def save_checkpoint(step, reason="periodic"):

    tmp_path = CHECKPOINT_PATH + ".tmp"

    checkpoint = {
        "step": step,
        "optimizer_step": step,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "reason": reason,

        "model_config": {
            "vocab_size": 32000,
            "d_model": 768,
            "num_layers": 12,
            "context_length": 4096,
        },
    }

    torch.save(
        checkpoint,
        tmp_path
    )

    os.replace(
        tmp_path,
        CHECKPOINT_PATH
    )

    size_gb = (
        os.path.getsize(CHECKPOINT_PATH)
        / 1024**3
    )

    print(
        f"✓ Checkpoint saved | "
        f"step={step} | "
        f"size={size_gb:.2f} GB | "
        f"reason={reason}"
    )


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

print()
print("=" * 60)
print("RWKV FP32 TRAINING")
print("=" * 60)

training_start = time.time()

for step in range(
    start_step + 1,
    MAX_STEPS + 1
):

    optimizer.zero_grad(
        set_to_none=True
    )

    step_loss = 0.0

    step_start = time.time()

    for micro in range(GRAD_ACCUM):

        batch = get_batch()

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        _, loss = model(
            input_ids=input_ids,
            labels=labels
        )

        loss_value = loss.item()

        step_loss += loss_value

        loss = loss / GRAD_ACCUM

        loss.backward()

        del input_ids
        del labels
        del batch
        del loss

    # --------------------------------------------------------
    # Gradient clipping
    # --------------------------------------------------------

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer.step()

    step_time = time.time() - step_start

    avg_loss = step_loss / GRAD_ACCUM

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    if (
        step == 1
        or step % 10 == 0
        or step == MAX_STEPS
    ):

        elapsed = time.time() - training_start

        print(
            f"step={step:4d} | "
            f"loss={avg_loss:.4f} | "
            f"grad={grad_norm.item():.3f} | "
            f"time={step_time:.2f}s | "
            f"elapsed={elapsed/60:.1f}m"
        )

    # --------------------------------------------------------
    # Rolling checkpoint every 100 steps
    # --------------------------------------------------------

    if step % 100 == 0:

        save_checkpoint(
            step,
            reason="periodic"
        )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    gc.collect()


# ------------------------------------------------------------
# Final checkpoint
# ------------------------------------------------------------

save_checkpoint(
    MAX_STEPS,
    reason="final"
)

total_time = time.time() - training_start

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(
    f"Final optimizer step: {MAX_STEPS}"
)
print(
    f"Training time: "
    f"{total_time/3600:.2f} hours"
)
print(
    f"Checkpoint: "
    f"{CHECKPOINT_PATH}"
)

In [ ]:
# ============================================================
# RWKV FINAL INFERENCE — LOCATE OFFICIAL TEST FILE
# ============================================================

from pathlib import Path
import json

candidates = list(Path("/kaggle/input").rglob("test.json"))

print("Found:")
for p in candidates:
    print(" ", p)

if not candidates:
    raise FileNotFoundError(
        "Could not find Phase1-Test.json under /kaggle/input"
    )

TEST_JSON = candidates[0]

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print()
print("Test file:", TEST_JSON)
print("Number of examples:", len(test_data))

first_id = next(iter(test_data))
print("First ID:", first_id)
print("Prompt preview:")
print(test_data[first_id]["text"][:300])

In [ ]:
# ============================================================
# LOAD FINAL RWKV CHECKPOINT
# ============================================================

from pathlib import Path
import torch

CHECKPOINT_PATH = Path(
    "/kaggle/working/phase1_rwkv/checkpoint_latest.pt"
)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}"
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint step:", checkpoint.get("step"))
print("Checkpoint keys:", list(checkpoint.keys()))

# Recreate model using the saved config if available
saved_config = checkpoint.get("model_config", checkpoint.get("config"))

print("\nSaved config:")
print(saved_config)

# ------------------------------------------------------------
# IMPORTANT:
# Use the SAME RWKV class/configuration that was used for
# training above this cell.
# ------------------------------------------------------------

model = RWKVLanguageModel(
    vocab_size=32000,
    context_length=4096,
    d_model=768,
    n_layer=12,
)

model.load_state_dict(checkpoint["model_state_dict"])

model = model.to(DEVICE)
model.eval()

print()
print("✓ Final RWKV checkpoint loaded")
print("Parameters:", sum(p.numel() for p in model.parameters()))